# **WhisperX Demo** 🎙️✨

Notebook creado por Carlos Santana Vega ([@DotCSV](https://twitter.com/DotCSV)) y GPT-4.

---

## ¿Cómo empezar? 🚀

1. **Habilita la GPU**: Para obtener una transcripción rápida, asegúrate de habilitar la GPU. Ve a "Runtime" > "Change runtime type" > y selecciona "GPU" en la opción "Hardware accelerator".
2. **Sube tu archivo de audio**: Usa la herramienta de abajo para subir tu archivo de audio.
3. **Ejecuta las celdas**: Simplemente ejecuta las celdas en orden, ¡y verás la magia suceder!

**Nota**: Si eres nuevo en Google Colab, cada celda con código se ejecuta haciendo clic en el botón de reproducción (▶️) a la izquierda de la celda, o puedes presionar `Shift + Enter`.

## Instalación del paquete 📦

Primero, necesitamos instalar WhisperX.

In [ ]:
!pip install git+https://github.com/m-bain/whisperx.git --upgrade
!pip install yt-dlp

  Cloning https://github.com/m-bain/whisperx.git to /tmp/pip-req-build-khi7_zag
  Running command git clone --filter=blob:none --quiet https://github.com/m-bain/whisperx.git /tmp/pip-req-build-khi7_zag
  Resolved https://github.com/m-bain/whisperx.git to commit 2cfd7b7c5c7bba144954364db747319b50e8232b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 119.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 64.4 MB/s eta 0:00:00
   ━━

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 75.4 MB/s eta 0:00:00


## Descarga el audio del video

In [ ]:
import yt_dlp

In [ ]:
yt_video_url = 'https://www.youtube.com/watch?v=2vv4hHAvqnE'

In [ ]:
def download_audio(url: str, out_dir: str = "audio") -> str:
    opts = {
        "format": "bestaudio/best",
        "outtmpl": f"{out_dir}/%(id)s.%(ext)s",
        "postprocessors": [{
            "key": "FFmpegExtractAudio",
            "preferredcodec": "wav",   # WhisperX trabaja cómodo con wav
        }],
        "postprocessor_args": ["-ar", "16000", "-ac", "1"],  # 16 kHz mono, lo que espera Whisper
        "quiet": True,
    }
    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(url, download=True)
        return f"{out_dir}/{info['id']}.wav"

audio_path = download_audio(yt_video_url)

## Transcripción de voz 🗣️

Ahora, vamos a transcribir tu archivo de audio. Puedes ajustar la `batch_size` y el `compute_type` según tus necesidades.

In [ ]:
import whisperx
import gc

device = "cuda"
batch_size = 16
compute_type = "float16"

# Transcripción
model = whisperx.load_model("large-v2", device, compute_type=compute_type)
audio = whisperx.load_audio(audio_path)
result = model.transcribe(audio, language="es", batch_size=batch_size)
print(result["segments"])

# Alineación
model_a, metadata = whisperx.load_align_model(language_code=result["language"], device=device)
result = whisperx.align(result["segments"], model_a, metadata, audio, device, return_char_alignments=False)
print(result["segments"])

2026-08-17 15:56:40 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-08-17 15:56:40 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`
/usr/local/lib/python3.12/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issue

[{'text': ' Bueno, Jordi, que tenemos que volver a... Estoy pidiendo un tip para ver si me abren. No, que volvemos a nuestra oficina de toda la vida. Pero antes era roja y ahora todo esto es negro, ¿no?', 'start': 0.503, 'end': 12.029, 'avg_logprob': -0.352556819265539}, {'text': ' ¿Qué tal? ¿Cómo estáis? ¿Todo bien? ¿Qué tal? ¿Qué tal? ¿Qué tal? ¿Qué tal? Le habéis gustado el color a esto, ¿eh? Esto tenía colores muy vivos antes y ahora lo habéis convertido en... Bueno, blanco y gris. Era muy bokeh. Era muy bokeh. Blanco, gris y marrón. Era alegre. Era alegre. Era muy alegre, era muy alegre. Es curioso entrar en tu casa, pero que ya no es tu casa. Ya, ya, ya. Es nuestra casa ahora. Sí, pero de otra manera. Antes era nuestra casa de verdad. Y ahora es nuestra casa compartida.', 'start': 18.104, 'end': 45.138, 'avg_logprob': -0.2929936427219658}, {'text': ' porque nos gusta superar la parte que puede estar contiene entre comillas', 'start': 48.192, 'end': 55.381, 'avg_logprob': -0.95955

100%|██████████| 360M/360M [00:01<00:00, 302MB/s]


2026-08-17 16:00:41 - whisperx.alignment - INFO - Downloading NLTK punkt_tab data for sentence splitting...
[{'start': 0.563, 'end': 4.045, 'text': ' Bueno, Jordi, que tenemos que volver a... Estoy pidiendo un tip para ver si me abren.', 'words': [{'word': 'Bueno,', 'start': 0.563, 'end': 0.783, 'score': np.float64(0.357)}, {'word': 'Jordi,', 'start': 0.803, 'end': 1.223, 'score': np.float64(0.457)}, {'word': 'que', 'start': 1.644, 'end': 1.744, 'score': np.float64(0.751)}, {'word': 'tenemos', 'start': 1.784, 'end': 2.044, 'score': np.float64(0.901)}, {'word': 'que', 'start': 2.084, 'end': 2.164, 'score': np.float64(0.889)}, {'word': 'volver', 'start': 2.224, 'end': 2.464, 'score': np.float64(0.946)}, {'word': 'a...', 'start': 2.504, 'end': 2.584, 'score': np.float64(0.876)}, {'word': 'Estoy', 'start': 2.664, 'end': 2.844, 'score': np.float64(0.454)}, {'word': 'pidiendo', 'start': 2.884, 'end': 3.084, 'score': np.float64(0.223)}, {'word': 'un', 'start': 3.124, 'end': 3.164, 'score': np

⬇️ Si quieres **descargar la transcripción** como un archivo de texto, ejecuta la siguiente celda.

In [ ]:
from google.colab import files

# Concatenating the transcribed segments
transcription_text = "\n".join([segment['text'] for segment in result["segments"]])

# Writing to a .txt file
with open('transcription.txt', 'w') as file:
    file.write(transcription_text)

# Download the .txt file
files.download('transcription.txt')

⬇️ Y si quieres **en .json la transcripción al completo** junto a sus timestamps, ejecuta la siguiente celda.

In [ ]:
import json

# Saving the entire result as a JSON file
with open('transcription.json', 'w') as file:
    json.dump(result, file, indent=4)

# Download the JSON file
files.download('transcription.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Diarizacion

In [ ]:
from whisperx.diarize import DiarizationPipeline
from google.colab import userdata;

HF_TOKEN = userdata.get('HF_TOKEN')

# 3. Assign speaker labels
diarize_model = DiarizationPipeline(token=HF_TOKEN, device=device)

# add min/max number of speakers if known
diarize_segments = diarize_model(audio, min_speakers=2)
result = whisperx.assign_word_speakers(diarize_segments, result)

2026-08-17 16:06:01 - whisperx.diarize - INFO - Loading diarization model: pyannote/speaker-diarization-community-1


config.yaml:   0%|          | 0.00/444 [00:00<?, ?B/s]

segmentation/pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 5.91MB            

segmentation/pytorch_model.bin: downloading bytes:           |  0.00B            

plda/xvec_transform.npz: reconstructing file:   0%|          |  0.00B /  134kB            

...

plda/plda.npz: reconstructing file:   0%|          |  0.00B /  134kB            

plda/plda.npz: downloading bytes:           |  0.00B            

embedding/pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 26.6MB            

embedding/pytorch_model.bin: downloading bytes:           |  0.00B            

/usr/local/lib/python3.12/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1839.)
  std = sequences.std(dim=-1, correction=1)


## Output

In [ ]:
result = whisperx.assign_word_speakers(diarize_segments, result)
print(diarize_segments)
print(result["segments"]) # segments are now assigned speaker IDs